In [0]:
import requests
# Parâmetros
API_KEY = "d65d7a05e58bdcccc8a44361df552e65"
cidade = "Curitiba"
# URL da API
url = f"http://api.openweathermap.org/data/2.5/weather?q={cidade}&appid={API_KEY}&units=metric&lang=pt_br"
# Requisição
response = requests.get(url)
data = response.json()

data

In [0]:
%sql
-- 1. Criar o catálogo
CREATE CATALOG IF NOT EXISTS projeto_API;

-- 2. Criar o schema dentro do catálogo projeto_API
CREATE SCHEMA IF NOT EXISTS projeto_API.pro_clima;

-- 3. Criar os volumes (usando Managed Volume para simplificar)
CREATE VOLUME IF NOT EXISTS projeto_API.pro_clima.bronze;
CREATE VOLUME IF NOT EXISTS projeto_API.pro_clima.silver;
CREATE VOLUME IF NOT EXISTS projeto_API.pro_clima.gold;

In [0]:
#lendo o arquivo e convertendo para pandas e depois para spark para gravar no delta 
import requests
import pandas as pd
from pyspark.sql import SparkSession

# Transformar em DataFrame Pandas
df_pandas = pd.DataFrame([data])

# Converter para Spark
df_bronze = spark.createDataFrame(df_pandas)

# Visualizar
display(df_bronze)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Definindo o caminho baseado no SQL que você executou
# Formato: /Volumes/CATALOGO/SCHEMA/VOLUME/NOME_ARQUIVO
bronze_path = "/Volumes/projeto_api/pro_clima/bronze"

# Cast nested struct fields to consistent types to avoid schema conflicts
df_bronze_typed = df_bronze.withColumn(
    "main",
    F.struct(
        F.col("main.feels_like").cast(DoubleType()).alias("feels_like"),
        F.col("main.grnd_level").cast("long").alias("grnd_level"),
        F.col("main.humidity").cast("long").alias("humidity"),
        F.col("main.pressure").cast("long").alias("pressure"),
        F.col("main.sea_level").cast("long").alias("sea_level"),
        F.col("main.temp").cast(DoubleType()).alias("temp"),
        F.col("main.temp_max").cast(DoubleType()).alias("temp_max"),
        F.col("main.temp_min").cast(DoubleType()).alias("temp_min")
    )
)

# Adicionamos metadados: data de processamento e nome do arquivo original
df_bronze_final = df_bronze_typed.withColumn("ingestion_timestamp", F.current_timestamp()) \
                           .withColumn("source_file", F.lit(f"weather_{cidade}"))

# Salvando - Usamos 'append' para acumular as consultas (Histórico)
df_bronze_final.write.format("delta").mode("append").option("mergeSchema", "true").save(bronze_path)

print(f"Dados inseridos na Bronze com sucesso em: {bronze_path}")

In [0]:
# Lendo da Bronze
df_silver_raw = spark.read.format("delta").load(bronze_path)

# Tratamento: Achatando o JSON e selecionando colunas úteis
df_silver = df_silver_raw.select(
    F.col("name").alias("cidade"),
    F.col("main.temp").cast("double").alias("temperatura"),
    F.col("main.humidity").cast("int").alias("umidade"),
    F.col("weather")[0]["description"].alias("condicao_tempo"),
    F.col("wind.speed").alias("velocidade_vento"),
   # Conversão correta
    F.from_utc_timestamp(
        F.to_timestamp(F.from_unixtime("dt")),
        "America/Sao_Paulo"
    ).alias("data_referencia"),

    F.current_timestamp().alias("ingestion_timestamp")
)

# Salvando na Silver
silver_path = "/Volumes/projeto_api/pro_clima/silver"
df_silver.write.format("delta").mode("append").save(silver_path)

display(df_silver)

In [0]:
df_silver_full = spark.read.format("delta").load(silver_path)

df_gold = (
    df_silver_full
    .drop("ingestion_timestamp")
    .dropDuplicates(["cidade", "data_referencia"])  # ajuste aqui
    .withColumn(
        "data_hora",
        F.from_utc_timestamp(F.current_timestamp(), "America/Sao_Paulo")
    )
)

gold_path = "/Volumes/projeto_api/pro_clima/gold"

df_gold.write.format("delta").mode("overwrite").save(gold_path)

display(df_gold)

In [0]:
# Nome da tabela no Unity Catalog
tb_dados_gold = "projeto_api.pro_clima.clima_consolidado"

# Salva o DataFrame como uma tabela Delta oficial
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tb_dados_gold)

print(f"Tabela {tb_dados_gold} criada com sucesso!")